# Reconia — LiLT trên DocILE / SROIE / CORD
Cùng OCR, bbox, nhãn và split với baseline. Không dùng ảnh. Smoke trước full; không hỗ trợ resume.
Chỉ số nội bộ, không phải điểm DocILE chính thức. SROIE chấm thêm bốn key gốc.


In [ ]:
DATASET = "docile"
RUN_MODE = "smoke"
RUN_ID = f"lilt-{DATASET}-seed42-ocr-v1"
EPOCHS = 20
SEED = 42
BACKBONE_REPO = "SCUT-DLVCLab/lilt-roberta-en-base"
BACKBONE_REVISION = "96328faae824978e47bd0e8ea09f66b85efda5f5"


In [ ]:
import os, sys, json, hashlib, shutil, subprocess, zipfile, re
from pathlib import Path
assert DATASET in {"docile", "sroie", "cord"}
assert RUN_MODE in {"smoke", "train"}
assert re.fullmatch(r"[a-zA-Z0-9_-]+", RUN_ID), "Use a simple run ID"
assert EPOCHS > 0
# Keep Kaggle's existing CUDA PyTorch, NumPy and pandas installations.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "python-dotenv==1.0.1", "transformers==4.57.6", "Pillow>=10.0"], check=True)
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
CODE = Path("/kaggle/temp/reconia-code") / RUN_ID
DATA = Path("/kaggle/temp/reconia-lilt") / RUN_ID
bundle_manifests = list(INPUT.rglob("bundle-manifest.json"))
assert len(bundle_manifests) == 1, "Attach exactly one LiLT OCR bundle manifest"
bundle_manifest = json.loads(bundle_manifests[0].read_text())
def extract_bundle(filename, destination):
    matches = list(INPUT.rglob(filename))
    assert len(matches) == 1, f"Attach exactly one Reconia bundle: expected {filename}, found {matches}"
    expected_archive = bundle_manifest[filename]
    assert matches[0].stat().st_size == expected_archive["bytes"], filename
    with matches[0].open("rb") as stream:
        assert hashlib.file_digest(stream, "sha256").hexdigest() == expected_archive["sha256"], filename
    if destination.exists() and any(destination.iterdir()):
        raise RuntimeError("Extraction directory already used. Restart the session or choose a new RUN_ID.")
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(matches[0]) as z:
        assert len(z.namelist()) == len(set(z.namelist())), "Duplicate archive paths"
        for name in z.namelist():
            assert (destination / name).resolve().is_relative_to(destination.resolve())
        z.extractall(destination)
extract_bundle("reconia-code.bin", CODE)
provenance = json.loads((CODE / "source-provenance.json").read_text())
for name, digest in provenance["files_sha256"].items():
    assert hashlib.sha256((CODE / name).read_bytes()).hexdigest() == digest, name
extract_bundle(f"{DATASET}-export.bin", DATA / "export")
os.chdir(CODE)
sys.path.insert(0, str(CODE))
os.environ[f"{DATASET.upper()}_EXPORT_DIR"] = str(DATA / "export")
from app.data.portable import ExportAdapter
from app.data.page_images import sha256
adapter = ExportAdapter(DATA / "export", DATASET)
counts = {s: len(adapter.document_ids(s)) for s in ("train", "val", "test")}
expected_counts = {"docile": {"train": 4680, "val": 500, "test": 500},
                   "sroie": {"train": 500, "val": 63, "test": 63},
                   "cord": {"train": 800, "val": 100, "test": 100}}
assert counts == expected_counts[DATASET], counts
if DATASET == "sroie":
    matches = list(INPUT.rglob("sroie-field-value-targets.json"))
    assert len(matches) == 1
    RAW_TARGETS = matches[0]
    assert sha256(RAW_TARGETS) == bundle_manifest[RAW_TARGETS.name]["sha256"]
print("Verified OCR export:", DATASET, counts)
del adapter
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONHASHSEED"] = str(SEED)
import torch
assert torch.cuda.is_available(), "Enable a GPU accelerator in Notebook Settings"
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)
from huggingface_hub import snapshot_download
MODEL = snapshot_download(repo_id=BACKBONE_REPO, revision=BACKBONE_REVISION,
                          cache_dir="/kaggle/temp/huggingface",
                          allow_patterns=["config.json", "model.safetensors", "tokenizer.json",
                                          "tokenizer_config.json", "special_tokens_map.json", "vocab.json", "merges.txt"])
print("MODE:", RUN_MODE.upper(), "| Backbone:", BACKBONE_REPO, BACKBONE_REVISION)


In [ ]:
from datetime import datetime, timezone
OUT = WORK / "reconia-runs" / (RUN_ID + ("-smoke" if RUN_MODE == "smoke" else ""))
CHECKPOINTS = OUT / "checkpoints"
REPORTS = OUT / "reports"
CACHE = Path("/kaggle/temp/reconia-graphs") / OUT.name
if any(CHECKPOINTS.glob("*.pt")):
    raise RuntimeError("This RUN_ID already has checkpoints. Choose a new RUN_ID.")
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
command = [sys.executable, "-u", "-m", "app.training.train_lilt",
           "--dataset", DATASET, "--device", "cuda", "--seed", str(SEED),
           "--epochs", str(EPOCHS if RUN_MODE == "train" else 1),
           "--lr", "0.00005", "--head-lr", "0.0002", "--batch-size", "2",
           "--batch-accum", "8", "--patience", "5",
           "--model-path", str(MODEL), "--save-dir", str(CHECKPOINTS), "--report-dir", str(REPORTS)]
if RUN_MODE == "smoke":
    command += ["--limit-train", "8", "--limit-val", "4"]
configuration = {"mode": RUN_MODE, "run_id": RUN_ID, "epochs": EPOCHS, "seed": SEED,
                 "counts": counts, "command": command,
                 "gpu": torch.cuda.get_device_name(0), "started_utc": datetime.now(timezone.utc).isoformat(),
                 "source": provenance, "dataset": DATASET, "dataset_checksums_sha256": sha256(DATA / "export/manifest.json")}
configuration.update(backbone_repo=BACKBONE_REPO, backbone_revision=BACKBONE_REVISION,
                     window_policy="page-layout-all-windows-first-subword-v1", image_input="none", ocr_input="dataset-provided")
configuration.update(data_transport="private-verified-ocr-bundle")
if DATASET == "sroie":
    configuration["raw_targets_sha256"] = sha256(RAW_TARGETS)
(OUT / "run-config.json").write_text(json.dumps(configuration, indent=2))
(OUT / "requirements-freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
print("Output:", OUT)
print("MODE:", RUN_MODE.upper(), "| Training documents:", counts["train"] if RUN_MODE == "train" else 8)
print("Launching:", " ".join(command))
# Train in a child process so stopping the notebook does not hide Python errors.
subprocess.run(command, check=True, cwd=CODE)


In [ ]:
if RUN_MODE == "smoke":
    # Reload weights in a fresh process; use validation, never test for debugging.
    subprocess.run([sys.executable, "-u", "-m", "app.evaluation.predict",
                    "--model", "lilt", "--dataset", DATASET, "--split", "val",
                    "--checkpoint", str(CHECKPOINTS / f"lilt_{DATASET}_best.pt"),
                    "--limit", "4", "--device", "cuda", "--report-dir", str(REPORTS)], check=True, cwd=CODE)
    print("SMOKE PASSED. For the full experiment set RUN_MODE = 'train' and Save & Run All.")
else:
    evidence = REPORTS / f"predictions/predictions_lilt_{DATASET}_test.metrics.json"
    metrics = json.loads(evidence.read_text())
    assert metrics["complete"] and metrics["num_documents"] == metrics["expected_num_documents"] == counts["test"]
    print("Full local test verified:", {k: metrics[k] for k in ("precision", "recall", "f1", "protocol")})
    print("OCR-derived field-value F1:", metrics["value_metrics"]["f1"])
if DATASET == "sroie":
    score_split = "val" if RUN_MODE == "smoke" else "test"
    suffix = "_sample4" if RUN_MODE == "smoke" else ""
    command = [sys.executable, "-u", "-m", "app.evaluation.sroie_values",
               "--predictions", str(REPORTS / f"predictions/predictions_lilt_sroie_{score_split}{suffix}.json"),
               "--targets", str(RAW_TARGETS), "--split", score_split,
               "--output", str(REPORTS / f"predictions/lilt_sroie_{score_split}{suffix}.four-field.metrics.json")]
    if RUN_MODE == "smoke":
        command += ["--limit", "4"]
    subprocess.run(command, check=True, cwd=CODE)
archive = shutil.make_archive(str(WORK / OUT.name), "zip", root_dir=OUT.parent, base_dir=OUT.name)
from IPython.display import FileLink, display
print("Download and retain both best/latest checkpoints, reports and run configuration.")
os.chdir(WORK)
display(FileLink(str(Path(archive).relative_to(WORK))))


## Sau khi chạy
Tải ZIP sau khi Kaggle publish xong; giữ best/latest, tokenizer, config và reports.
Smoke chỉ kiểm tra đường chạy; chưa xác nhận chất lượng. Dùng RUN_MODE = "train" cho full sau khi smoke qua.
